# DJA × JWST multiple surveys crossmatch — visualization
Per-survey sky positions, separation distributions, the 2-D astrometric offset,
and the multi-survey / repeated-object situation.
Reads `data/crossmatched/dja_x_f150w.fits` (built by `build_crossmatch.py`).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from collections import Counter

plt.rcParams.update({
    "axes.grid": True, "grid.alpha": 0.25, "axes.edgecolor": "#888",
    "axes.linewidth": 0.8, "figure.dpi": 110, "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
ROOT = os.path.expanduser("~/ssl_outthere")
IMG  = os.path.join(ROOT, "data/image")
FILTER = "f150w"
SURVEYS = ["cosmos", "ceers", "outthere", "jades"]
# Okabe-Ito categorical palette (colourblind-safe), fixed per survey
CLR = {"cosmos": "#0072B2", "ceers": "#E69F00", "outthere": "#009E73", "jades": "#CC79A7"}

t = Table.read(os.path.join(ROOT, "data/crossmatched/dja_x_f150w.fits"))
survey = np.char.strip(np.asarray(t["survey"]).astype(str))
dja_id = np.asarray(t["dja_id"]); ra = np.asarray(t["ra"], float); dec = np.asarray(t["dec"], float)
sep = np.asarray(t["sep_arcsec"], float)
rel = np.asarray(t["rel_path"]).astype(str); loc = np.asarray(t["local_idx"])
print(f"{len(t)} matched rows | unique spectra {len(np.unique(dja_id))}")
for s in SURVEYS:
    print(f"  {s:9s} {int((survey==s).sum()):6d}")

# Recover the IMAGE ra/dec per matched row by joining image_index on (rel_path, local_idx)
img_ra = np.full(len(t), np.nan); img_dec = np.full(len(t), np.nan)
for s in SURVEYS:
    it = Table.read(os.path.join(IMG, f"image_index_{s}_{FILTER}.fits"))
    key = {(rp, int(li)): (r, d) for rp, li, r, d in
           zip(np.asarray(it["rel_path"]).astype(str), np.asarray(it["local_idx"]),
               np.asarray(it["ra"], float), np.asarray(it["dec"], float))}
    for i in np.where(survey == s)[0]:
        v = key.get((rel[i], int(loc[i])))
        if v: img_ra[i], img_dec[i] = v
# offsets in arcsec (image - DJA)
dra  = (img_ra - ra) * np.cos(np.radians(dec)) * 3600.0
ddec = (img_dec - dec) * 3600.0
print("offset recovered for", int(np.isfinite(dra).sum()), "/", len(t), "rows")

## 1. Sky positions per survey
Small multiples (each survey's footprint) + a combined all-sky panel.

In [ ]:
fig, axes = plt.subplots(1, len(SURVEYS) + 1, figsize=(3.1*(len(SURVEYS)+1), 3.1))
for ax, s in zip(axes[:-1], SURVEYS):
    m = survey == s
    ax.scatter(ra[m], dec[m], s=4, c=CLR[s], alpha=0.5, edgecolors="none")
    ax.set_title(f"{s}  (n={int(m.sum())})"); ax.set_xlabel("RA [deg]"); ax.invert_xaxis()
axes[0].set_ylabel("Dec [deg]")
axc = axes[-1]
for s in SURVEYS:
    m = survey == s
    axc.scatter(ra[m], dec[m], s=4, c=CLR[s], alpha=0.5, edgecolors="none", label=s)
axc.set_title("all surveys"); axc.set_xlabel("RA [deg]"); axc.invert_xaxis()
axc.legend(markerscale=2, fontsize=8, loc="best")
fig.tight_layout(); plt.show()

## 1b. outthere by subfield
outthere's cutouts span several widely-separated fields (`tile`), so it collapses to dots in the all-sky panel above. Here it is split by subfield, with co-located matches from the other surveys overlaid — showing which deep field each outthere subfield lands on (and thus the source of the multi-survey overlaps).

In [ ]:
# outthere lives in several widely-separated subfields (`tile`), so it collapses in
# the all-sky panel above. Facet by subfield, overlaying co-located matches from the
# other surveys to show which deep field each outthere subfield sits on.
tile = np.char.strip(np.asarray(t["tile"]).astype(str))
mo = survey == "outthere"
tl_u, tl_c = np.unique(tile[mo], return_counts=True)
tiles = tl_u[np.argsort(tl_c)[::-1]]
fig, axes = plt.subplots(1, len(tiles), figsize=(3.4*len(tiles), 3.3))
axes = np.atleast_1d(axes)
for ax, tl in zip(axes, tiles):
    sub = mo & (tile == tl)
    pad = 0.03
    r0, r1 = ra[sub].min()-pad, ra[sub].max()+pad
    d0, d1 = dec[sub].min()-pad, dec[sub].max()+pad
    win = (ra >= r0) & (ra <= r1) & (dec >= d0) & (dec <= d1)
    for s in SURVEYS:                       # co-located other-survey matches (overlap context)
        mm = win & (survey == s) & ~sub
        if mm.any():
            ax.scatter(ra[mm], dec[mm], s=10, c=CLR[s], alpha=0.45, edgecolors="none", label=s)
    ax.scatter(ra[sub], dec[sub], s=16, c=CLR["outthere"], edgecolors="k", linewidths=0.2,
               label=f"outthere/{tl}")
    ax.set_title(f"{tl}  (outthere n={int(sub.sum())})"); ax.set_xlabel("RA [deg]"); ax.invert_xaxis()
    ax.legend(fontsize=7, markerscale=1.3, loc="best")
axes[0].set_ylabel("Dec [deg]")
fig.suptitle("outthere subfields (green) + co-located matches from other surveys",
             y=1.03, fontweight="bold")
fig.tight_layout(); plt.show()

## 2. Separation distribution
Match quality per survey (arcsec).

In [ ]:
# Separation distribution. Top row: the 3 whole surveys. Bottom row: outthere by subfield
# (distinct hues, not one colour family).
tile = np.char.strip(np.asarray(t["tile"]).astype(str))
o_tiles = [tl for tl, _ in Counter(tile[survey == "outthere"]).most_common()]
o_colors = ["#009E73", "#D55E00", "#56B4E9", "#9467bd"]   # green / vermillion / sky-blue / purple
bins = np.linspace(0, 0.5, 101)

def _sep_row(ax_h, ax_c, groups, title):
    for lab, m, c in groups:
        ax_h.hist(sep[m], bins=bins, histtype="step", lw=1.8, color=c, density=True, alpha=0.8,
                  label=f"{lab} (n={int(m.sum())}, med {np.median(sep[m]):.3f}″)")
        x = np.sort(sep[m]); ax_c.plot(x, np.arange(1, len(x)+1)/len(x), lw=1.6, color=c, label=lab)
    ax_h.set_ylabel("density"); ax_h.set_title(f"{title} — histogram"); ax_h.legend(fontsize=7)
    ax_c.axhline(0.95, ls=":", c="#888", lw=1); ax_c.set_ylabel("cumulative fraction")
    ax_c.set_title(f"{title} — cumulative"); ax_c.legend(fontsize=7)
    for a in (ax_h, ax_c): a.set_xlabel("separation [arcsec]")

fig, axes = plt.subplots(2, 2, figsize=(11, 6.4))
_sep_row(axes[0, 0], axes[0, 1], [(s, survey == s, CLR[s]) for s in ["cosmos", "ceers", "jades"]], "surveys")
_sep_row(axes[1, 0], axes[1, 1],
         [(f"outthere/{tl}", (survey == "outthere") & (tile == tl), c) for tl, c in zip(o_tiles, o_colors)],
         "outthere by subfield")
fig.tight_layout(); plt.show()

## 3. 2-D astrometric offset
(image − DJA) position offset in arcsec. A blob centred on (0,0) = no systematic; an off-centre blob = an astrometric shift between the spectrum and image catalogues.

In [ ]:
# 2-D offset, grid-wrapped (not one long row). outthere subfields in distinct hues; density panel last.
tile = np.char.strip(np.asarray(t["tile"]).astype(str))
o_tiles = [tl for tl, _ in Counter(tile[survey == "outthere"]).most_common()]
o_colors = ["#009E73", "#D55E00", "#56B4E9", "#9467bd"]   # green / vermillion / sky-blue / purple
groups = [(s, survey == s, CLR[s]) for s in ["cosmos", "ceers", "jades"]]
groups += [(f"outthere/{tl}", (survey == "outthere") & (tile == tl), c)
           for tl, c in zip(o_tiles, o_colors)]

lim = 0.4
ncols = 4
npan  = len(groups) + 1                         # + the 'all (density)' panel
nrows = int(np.ceil(npan / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 3.2*nrows))
axes = np.atleast_1d(axes).ravel()
for ax, (lab, m0, c) in zip(axes, groups):
    m = m0 & np.isfinite(dra)
    ax.scatter(dra[m], ddec[m], s=6, c=c, alpha=0.45, edgecolors="none")
    ax.plot(np.median(dra[m]), np.median(ddec[m]), "k+", ms=9, mew=2)
    ax.axhline(0, c="#bbb", lw=0.8); ax.axvline(0, c="#bbb", lw=0.8)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_title(f"{lab} (n={int(m.sum())})\nmed=({np.median(dra[m]):+.03f},{np.median(ddec[m]):+.03f})″", fontsize=8)
    ax.set_xlabel("ΔRA·cosδ [″]"); ax.set_ylabel("ΔDec [″]")
# 'all' density panel, placed last
axc = axes[len(groups)]; mm = np.isfinite(dra)
hb = axc.hexbin(dra[mm], ddec[mm], gridsize=40, extent=(-lim, lim, -lim, lim), cmap="magma", mincnt=1)
axc.axhline(0, c="w", lw=0.6); axc.axvline(0, c="w", lw=0.6); axc.set_aspect("equal")
axc.set_title("all (density)", fontsize=8); axc.set_xlabel("ΔRA·cosδ [″]")
fig.colorbar(hb, ax=axc, shrink=0.8, label="count")
for ax in axes[npan:]:      # hide any unused grid cells
    ax.axis("off")
fig.tight_layout(); plt.show()

## 4. Repeated objects
Two kinds of repetition:
- **multi-survey spectra** — one DJA spectrum matched in ≥2 survey footprints (overlap);
- **reused cutouts** — one image cutout matched by ≥2 different DJA spectra.

In [ ]:
# (a) multiplicity of each spectrum across surveys
uid, inv = np.unique(dja_id, return_inverse=True)
mult = np.bincount(inv)
print("spectra by #survey-matches:", dict(Counter(mult.tolist())))
dup_ids = uid[mult >= 2]
print(f"multi-survey spectra: {len(dup_ids)}")

by_id = {}
for did, s in zip(dja_id, survey):
    by_id.setdefault(int(did), set()).add(s)
combo = Counter(tuple(sorted(by_id[int(d)])) for d in dup_ids)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
items = combo.most_common()
labels = [" + ".join(k) for k, _ in items]; vals = [v for _, v in items]
ax[0].barh(range(len(vals)), vals, color="#555")
ax[0].set_yticks(range(len(vals))); ax[0].set_yticklabels(labels, fontsize=8); ax[0].invert_yaxis()
ax[0].set_xlabel("# spectra"); ax[0].set_title(f"multi-survey overlap combos (n={len(dup_ids)})")
for i, v in enumerate(vals): ax[0].text(v, i, f" {v}", va="center", fontsize=8)

ax[1].scatter(ra, dec, s=3, c="#ddd", edgecolors="none")
dm = np.isin(dja_id, dup_ids)
for s in SURVEYS:
    mm = dm & (survey == s)
    ax[1].scatter(ra[mm], dec[mm], s=10, c=CLR[s], edgecolors="none", label=s)
ax[1].set_xlabel("RA [deg]"); ax[1].set_ylabel("Dec [deg]"); ax[1].invert_xaxis()
ax[1].set_title("multi-survey spectra on sky"); ax[1].legend(fontsize=8, markerscale=1.5)
fig.tight_layout(); plt.show()

In [ ]:
# (b) reused cutouts: same (survey, rel_path, local_idx) matched by >1 spectrum
cut_key = np.array([f"{s}|{rp}|{int(li)}" for s, rp, li in zip(survey, rel, loc)])
uk, ik = np.unique(cut_key, return_inverse=True)
cmult = np.bincount(ik)
print(f"cutouts matched by >=2 spectra: {int((cmult>=2).sum())} / {len(uk)} unique cutouts")
print("cutout reuse multiplicity:", dict(Counter(cmult.tolist())))
for s in SURVEYS:
    m = survey == s
    ks, cc = np.unique(cut_key[m], return_counts=True)
    print(f"  {s:9s}: {int((cc>=2).sum()):4d} reused / {len(ks):5d} cutouts  ({100*(cc>=2).mean():.1f}%)")